# Notebook 06: Model Inversion Attacks

## Why This Matters
Membership inference asks: "was this specific record in the training set?" Model inversion asks a different and in some ways more alarming question: "what does a *representative* training example look like, given only model access?"

Fredrikson et al. (2015) demonstrated this against a medical model that predicted warfarin dosage — they recovered recognizable face images from a model trained on medical records. The implication: deploying a model can reveal information about your training data without sharing the data itself.

## Structure
1. What is model inversion and why it works
2. Fredrikson et al.'s original attack (gradient-based inversion)
3. GMI: Generative Model Inversion with GAN priors (Zhang et al., 2020)
4. White-box vs black-box inversion
5. Evaluating inversion quality
6. Defenses: output perturbation, prediction APIs, DP

## What You'll Learn
- Why class-conditional gradients reveal training data structure
- How GAN priors dramatically improve inversion quality
- The difference between recovering a *specific* record vs a *representative* one
- Why this attack is distinct from membership inference but related
- What output API design choices reduce inversion risk

## Background

### The two flavors of model inversion

Model inversion is not one attack but a family, and it's important to distinguish two different goals:

**Class representative inversion**: recover an input that represents class $c$ according to the model. This doesn't recover any specific training example — it recovers what the model has learned to associate with the class. This is similar in spirit to maximizing a class activation in a network visualization. The result is a synthetic image that the model classifies with high confidence.

**Training data reconstruction**: recover something close to a *specific* training record. This is harder and requires stronger assumptions (e.g., white-box access, or a very overfit model). Large language models memorize training text verbatim and can produce it through sampling — Carlini et al. (2021) demonstrated this at scale for GPT-2.

Most practical model inversion attacks (Fredrikson et al., Zhang et al.) target class representative recovery, not specific record reconstruction. The distinction matters for threat modeling: class representative recovery leaks information about what *class* training data looks like, not about specific individuals.

### Fredrikson et al. (2015): the first practical model inversion

Fredrikson, Jha & Ristenpart (2015) — *"Model Inversion Attacks that Exploit Confidence Information and Basic Countermeasures"* (CCS 2015) — demonstrated two inversion scenarios:

1. **Pharmacogenetics model**: a linear model predicting warfarin dosage. The model returned exact dosage predictions. By solving an optimization problem over inputs given the target prediction, they recovered input feature values.

2. **Face recognition model**: a neural network for facial recognition. Using the model's confidence scores, they recovered images that were recognizable as the training subjects. This was the alarming result — visible face images reconstructed from model outputs.

The attack works by gradient ascent on the input: starting from random noise or a neutral image, update the input to maximize the model's confidence for class $c$. If the model has learned to associate class $c$ with specific visual features, gradient ascent toward class $c$ will move the input toward those features.

### GMI: GAN-based model inversion (Zhang et al., 2020)

The quality of naive gradient ascent is limited: the inverted image needs to look realistic (be on the training data manifold), but gradient ascent from random noise can produce adversarial-looking images that fool the classifier without resembling training data.

Zhang et al. (2020) — *"The Secret Revealer: Generative Model-Inversion Attacks Against Deep Neural Networks"* (CVPR 2020) — addressed this by incorporating a GAN as a prior over the input space. Instead of optimizing directly in pixel space, they optimize in the GAN's latent space:

$$z^* = \arg\max_z \log f_c(G(z)) - \lambda \cdot \mathcal{D}(G(z))$$

where $G$ is the GAN generator, $f_c$ is the target model's confidence for class $c$, and $\mathcal{D}$ is a discriminator that keeps the output on the image manifold. Searching in latent space rather than pixel space ensures the recovered image looks realistic.

### Why confidence scores are necessary (and dangerous)

Both attacks require the model to return confidence scores (softmax probabilities), not just a class label. The gradient of confidence with respect to the input is what drives the optimization. Models that return only the top-1 prediction are significantly harder to invert — but still not immune to black-box attacks that estimate gradients through repeated queries.

In [ ]:
%pip install -q torch torchvision numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(f"Device: {device}")

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])
train_data = torchvision.datasets.MNIST('./data', train=True,  download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_data, batch_size=256, shuffle=True)
test_data    = torchvision.datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader  = torch.utils.data.DataLoader(test_data, batch_size=256, shuffle=False)

In [ ]:
class MnistCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)
        self.drop  = nn.Dropout(0.1)  # low dropout → more memorization

    def forward(self, x):
        x = F.relu(self.conv1(x)); x = self.pool(x)
        x = F.relu(self.conv2(x)); x = self.pool(x)
        x = F.relu(self.fc1(x.flatten(1))); x = self.drop(x)
        return self.fc2(x)


# Train target model
target_model = MnistCNN().to(device)
opt = torch.optim.Adam(target_model.parameters(), lr=1e-3)

print("Training target model...")
for epoch in range(8):
    target_model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad(); F.cross_entropy(target_model(x), y).backward(); opt.step()

correct = total = 0
target_model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (target_model(x).argmax(1) == y).sum().item(); total += y.size(0)
print(f"Target model test accuracy: {correct/total:.4f}")

## 1. Gradient-Based Class Inversion (Fredrikson et al.)

The algorithm:
1. Start from a random or neutral input $x_0$
2. Iterate: $x_{t+1} = x_t + \alpha \cdot \nabla_x \log f_c(x_t) - \beta \cdot \nabla_x R(x_t)$
   - First term: gradient ascent toward class $c$ confidence
   - Second term: regularization $R$ to keep the image realistic (TV loss, L2 norm)
3. Output $x^* = x_T$ — the inverted class representative

This is conceptually identical to generating adversarial examples, except the *target* is a specific class rather than the wrong class, and we care about visual realism rather than just misclassification.

In [ ]:
def total_variation_loss(x: torch.Tensor) -> torch.Tensor:
    """TV regularization: penalize pixel-level discontinuities."""
    diff_h = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().sum()
    diff_w = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().sum()
    return diff_h + diff_w


def gradient_inversion(
    model: nn.Module,
    target_class: int,
    n_steps: int = 500,
    lr: float = 0.05,
    tv_weight: float = 1e-4,
    l2_weight: float = 1e-5,
    input_shape: tuple = (1, 1, 28, 28),
    n_restarts: int = 3,
) -> torch.Tensor:
    """
    Gradient-based model inversion (Fredrikson et al., 2015).

    Maximizes model confidence for target_class via gradient ascent.
    TV + L2 regularization keeps result visually plausible.

    Returns the best inverted image across all restarts.
    """
    model.eval()
    best_img  = None
    best_conf = -1.0

    for restart in range(n_restarts):
        # Random initialization in normalized space
        x = torch.randn(input_shape, device=device, requires_grad=True)
        optimizer = torch.optim.Adam([x], lr=lr)

        for step in range(n_steps):
            optimizer.zero_grad()
            logits = model(x)
            conf_loss = -F.log_softmax(logits, dim=1)[0, target_class]  # maximize log-confidence
            reg_loss  = tv_weight * total_variation_loss(x) + l2_weight * x.pow(2).sum()
            loss = conf_loss + reg_loss
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            conf = F.softmax(model(x), dim=1)[0, target_class].item()
        if conf > best_conf:
            best_conf = conf
            best_img  = x.detach().clone()

    return best_img, best_conf


# Invert all 10 digit classes
print("Inverting all 10 MNIST classes via gradient ascent...")
inverted_images = []
confidences = []

for digit in range(10):
    img, conf = gradient_inversion(target_model, digit, n_steps=300, n_restarts=3)
    inverted_images.append(img)
    confidences.append(conf)
    print(f"  Digit {digit}: confidence = {conf:.4f}")

In [ ]:
# Visualize inverted images vs real training examples
def unnorm(x):
    return np.clip((x.cpu().squeeze().numpy() * 0.3081 + 0.1307), 0, 1)

# Get one real example per digit
real_examples = {}
for x, y in train_data:
    label = y if isinstance(y, int) else y.item()
    if label not in real_examples:
        real_examples[label] = x
    if len(real_examples) == 10:
        break

fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle("Model Inversion: Recovered Class Representatives vs Real Training Examples", fontsize=11)

for digit in range(10):
    # Top row: inverted images
    axes[0, digit].imshow(unnorm(inverted_images[digit]), cmap='gray', vmin=0, vmax=1)
    axes[0, digit].set_title(f'{digit}\n{confidences[digit]:.2f}', fontsize=8)
    axes[0, digit].axis('off')

    # Bottom row: real examples
    axes[1, digit].imshow(unnorm(real_examples[digit]), cmap='gray', vmin=0, vmax=1)
    axes[1, digit].axis('off')

axes[0, 0].set_ylabel('Inverted', fontsize=9)
axes[1, 0].set_ylabel('Real', fontsize=9)

plt.tight_layout()
plt.savefig('model_inversion_results.png', bbox_inches='tight')
plt.show()

print("Top row: images recovered by gradient ascent — what the model 'thinks' each digit looks like")
print("Bottom row: real training examples")
print("The inverted images capture the class structure learned by the model.")

## 2. Inversion with a Learned Prior (GMI-style)

Pure gradient ascent in pixel space produces images that maximize classifier confidence but may not look realistic. A learned prior forces the search to stay on the data manifold.

The simplest learned prior for MNIST: train a small decoder that maps class-conditioned latent vectors to realistic digits. Optimize in latent space instead of pixel space — every point in latent space decodes to a realistic-looking image, so the search stays on the manifold.

This is a simplified version of GMI (Zhang et al., 2020). The full GMI uses a separately trained GAN — we use a class-conditional VAE decoder for efficiency.

In [ ]:
class ConditionalDecoder(nn.Module):
    """Small conditional decoder: maps (latent_z, class) → image."""
    def __init__(self, latent_dim=32, n_classes=10):
        super().__init__()
        self.embed = nn.Embedding(n_classes, 16)
        self.net = nn.Sequential(
            nn.Linear(latent_dim + 16, 256), nn.ReLU(),
            nn.Linear(256, 512),             nn.ReLU(),
            nn.Linear(512, 28 * 28),
        )
        self.latent_dim = latent_dim

    def forward(self, z: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
        c_emb = self.embed(c)
        return self.net(torch.cat([z, c_emb], dim=1)).view(-1, 1, 28, 28)


# Train the decoder as a generative prior over MNIST
LATENT_DIM = 32
decoder = ConditionalDecoder(latent_dim=LATENT_DIM).to(device)
opt_dec = torch.optim.Adam(decoder.parameters(), lr=1e-3)

print("Training conditional decoder as image prior...")
for epoch in range(10):
    decoder.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        # Encode via random latent + reconstruct
        z = torch.randn(x.size(0), LATENT_DIM, device=device)
        x_hat = decoder(z, y)
        # Train to make decoder outputs distinguishable per class (rough prior)
        # Use the TARGET model to guide: decoder output should fool the target
        loss = F.cross_entropy(target_model(x_hat), y) * 0.5
        # Plus pixel reconstruction loss from real examples
        z_enc = torch.randn_like(z)
        x_hat2 = decoder(z_enc, y)
        loss += F.mse_loss(x_hat2, x) * 2.0
        opt_dec.zero_grad(); loss.backward(); opt_dec.step()
print("Decoder trained.")

In [ ]:
def latent_space_inversion(
    target_model: nn.Module,
    decoder: nn.Module,
    target_class: int,
    n_steps: int = 300,
    lr: float = 0.05,
    n_restarts: int = 5,
) -> tuple:
    """
    GMI-style inversion: optimize in latent space to stay on image manifold.

    Every decoded image looks realistic (it's generated by the prior).
    We search for the latent vector that maximizes target class confidence.
    """
    target_model.eval()
    decoder.eval()

    best_img  = None
    best_conf = -1.0
    y_target  = torch.tensor([target_class], device=device)

    for _ in range(n_restarts):
        z = torch.randn(1, LATENT_DIM, device=device, requires_grad=True)
        opt_z = torch.optim.Adam([z], lr=lr)

        for step in range(n_steps):
            opt_z.zero_grad()
            x_gen = decoder(z, y_target)
            conf_loss = -F.log_softmax(target_model(x_gen), dim=1)[0, target_class]
            conf_loss.backward()
            opt_z.step()

        with torch.no_grad():
            x_gen = decoder(z, y_target)
            conf  = F.softmax(target_model(x_gen), dim=1)[0, target_class].item()

        if conf > best_conf:
            best_conf = conf
            best_img  = x_gen.detach().clone()

    return best_img, best_conf


print("Running latent-space (GMI-style) inversion...")
latent_inverted = []
latent_confs = []
for digit in range(10):
    img, conf = latent_space_inversion(target_model, decoder, digit, n_steps=200, n_restarts=5)
    latent_inverted.append(img)
    latent_confs.append(conf)
    print(f"  Digit {digit}: confidence = {conf:.4f}")

In [ ]:
# Compare pixel-space vs latent-space inversion
fig, axes = plt.subplots(3, 10, figsize=(15, 5))
fig.suptitle("Inversion Comparison: Pixel-Space vs Latent-Space (GMI-style)", fontsize=11)

for digit in range(10):
    # Row 0: pixel-space gradient inversion
    axes[0, digit].imshow(unnorm(inverted_images[digit]), cmap='gray')
    axes[0, digit].set_title(f'{digit}', fontsize=8)
    axes[0, digit].axis('off')

    # Row 1: latent-space inversion
    axes[1, digit].imshow(unnorm(latent_inverted[digit]), cmap='gray')
    axes[1, digit].axis('off')

    # Row 2: real example
    axes[2, digit].imshow(unnorm(real_examples[digit]), cmap='gray')
    axes[2, digit].axis('off')

axes[0, 0].set_ylabel('Pixel-space\nGrad ascent', fontsize=8)
axes[1, 0].set_ylabel('Latent-space\n(GMI-style)', fontsize=8)
axes[2, 0].set_ylabel('Real\nexample', fontsize=8)

plt.tight_layout()
plt.savefig('inversion_comparison.png', bbox_inches='tight')
plt.show()

print("Pixel-space: maximizes confidence but may look unrealistic")
print("Latent-space: constrained to stay on image manifold → more realistic but sometimes less confident")

## 3. Black-Box Inversion via Finite Differences

White-box inversion requires gradient access. Black-box inversion estimates the gradient through repeated queries:

$$\frac{\partial f_c(x)}{\partial x_i} \approx \frac{f_c(x + h e_i) - f_c(x - h e_i)}{2h}$$

This requires $2d$ queries per gradient estimate for $d$-dimensional input. For 28×28 MNIST images that's 1568 queries per step — expensive but feasible for a motivated attacker with API access.

Natural evolution strategies (NES) provide a more efficient alternative: estimate the gradient using a population of noise-perturbed inputs, requiring far fewer queries than finite differences.

In [ ]:
def nes_gradient_estimate(
    query_fn,    # black-box function: input → confidence for target class
    x: np.ndarray,
    sigma: float = 0.1,
    n_samples: int = 50,
) -> np.ndarray:
    """
    Natural Evolution Strategies gradient estimate.
    Uses n_samples perturbations to estimate the gradient with far fewer queries
    than finite differences (n_samples << 2*d queries for finite diff).
    """
    grad = np.zeros_like(x)
    for _ in range(n_samples // 2):
        delta = np.random.randn(*x.shape).astype(np.float32)
        conf_pos = query_fn(x + sigma * delta)
        conf_neg = query_fn(x - sigma * delta)
        grad += (conf_pos - conf_neg) * delta
    return grad / (2 * sigma * n_samples)


def blackbox_inversion(
    model: nn.Module,
    target_class: int,
    n_steps: int = 100,
    lr: float = 0.1,
    n_samples: int = 20,
):
    """Black-box model inversion using NES gradient estimates."""
    model.eval()
    query_count = [0]

    def query(x_np: np.ndarray) -> float:
        query_count[0] += 1
        x_t = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            return F.softmax(model(x_t), dim=1)[0, target_class].item()

    x = np.random.randn(1, 28, 28).astype(np.float32) * 0.1

    for step in range(n_steps):
        grad = nes_gradient_estimate(query, x, sigma=0.1, n_samples=n_samples)
        x = x + lr * grad
        x = np.clip(x, -3, 3)

    final_conf = query(x)
    return torch.tensor(x, device=device).unsqueeze(0), final_conf, query_count[0]


# Demonstrate black-box inversion on digit 3
print("Black-box inversion for digit 3 (NES gradient estimates)...")
bb_img, bb_conf, n_queries = blackbox_inversion(target_model, target_class=3, n_steps=80, n_samples=20)

with torch.no_grad():
    wb_conf = F.softmax(target_model(inverted_images[3]), dim=1)[0, 3].item()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(unnorm(inverted_images[3]),   cmap='gray'); axes[0].set_title(f'White-box\nconf={wb_conf:.3f}'); axes[0].axis('off')
axes[1].imshow(unnorm(bb_img),               cmap='gray'); axes[1].set_title(f'Black-box (NES)\nconf={bb_conf:.3f}\n{n_queries} queries'); axes[1].axis('off')
axes[2].imshow(unnorm(real_examples[3]),     cmap='gray'); axes[2].set_title('Real example'); axes[2].axis('off')
fig.suptitle("Digit 3: White-box vs Black-box Inversion")
plt.tight_layout(); plt.savefig('blackbox_inversion.png', bbox_inches='tight'); plt.show()

print(f"White-box: {wb_conf:.4f} confidence (gradient access)")
print(f"Black-box: {bb_conf:.4f} confidence ({n_queries} queries, NES estimation)")
print("Black-box inversion is weaker but works without any model access beyond API queries")

## 4. Defenses Against Model Inversion

**API design defenses (easiest):**
- Return only the top-1 predicted class, no confidence scores → makes gradient computation impossible
- Return top-k without exact probabilities → degrades but doesn't eliminate black-box attacks
- Add small random noise to confidence scores → disrupts gradient-based inversion, raises query count needed

**Rate limiting:**
- Gradient-based inversion requires many queries (hundreds to thousands per image)
- NES black-box requires `2 × n_samples × n_steps` queries
- Rate limiting and query monitoring can detect and block inversion attempts

**Model-level defenses:**
- **Differential privacy training**: adds calibrated noise during training, provably limits what gradients reveal
- **Knowledge distillation to a privacy-preserving student**: train a student model that's constrained to not over-fit class-specific features

**Important caveat:**
No defense against model inversion provides the same formal guarantee that DP provides against membership inference. Class representative inversion (what we demonstrated here) is less directly harmful than record-level reconstruction, but still reveals what the model has learned about your data distribution.

In [ ]:
# Demonstrate: confidence score perturbation as a defense
def noisy_query_fn(model, x_t, target_class, noise_std=0.05):
    """API with output perturbation — adds noise to confidence scores."""
    model.eval()
    with torch.no_grad():
        probs = F.softmax(model(x_t), dim=1)
        noise = torch.randn_like(probs) * noise_std
        noisy_probs = (probs + noise).clamp(0, 1)
        noisy_probs = noisy_probs / noisy_probs.sum(dim=1, keepdim=True)  # renormalize
    return noisy_probs[0, target_class].item()


def blackbox_inversion_defended(
    model: nn.Module,
    target_class: int,
    noise_std: float,
    n_steps: int = 80,
    n_samples: int = 20,
):
    """Black-box inversion against a noisy API."""
    model.eval()
    query_count = [0]

    def query(x_np):
        query_count[0] += 1
        x_t = torch.tensor(x_np, dtype=torch.float32, device=device).unsqueeze(0)
        return noisy_query_fn(model, x_t, target_class, noise_std)

    x = np.random.randn(1, 28, 28).astype(np.float32) * 0.1
    for _ in range(n_steps):
        grad = nes_gradient_estimate(query, x, sigma=0.1, n_samples=n_samples)
        x = np.clip(x + 0.1 * grad, -3, 3)

    # Evaluate true confidence (without noise, as ground truth)
    x_t = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        true_conf = F.softmax(model(x_t), dim=1)[0, target_class].item()
    return x_t, true_conf, query_count[0]


print("Effect of output noise defense on black-box inversion (digit 7):")
print(f"{'Noise σ':>10} {'True conf (final)':>20} {'Queries used':>14}")
print("-" * 48)

for noise in [0.0, 0.02, 0.05, 0.10]:
    img_d, conf_d, queries_d = blackbox_inversion_defended(
        target_model, target_class=7, noise_std=noise, n_steps=80
    )
    print(f"{noise:>10.2f} {conf_d:>20.4f} {queries_d:>14}")

print()
print("Higher noise → lower final confidence → less informative inversion")
print("But: attacker can increase n_samples to average out noise (at query cost)")

## Summary

**Model inversion in one line:**  
Optimize an input to maximize model confidence for a target class — the gradient reveals what the model has learned to associate with that class.

**Key distinctions:**
- **Class representative** recovery (what we built): recovers what the class looks like to the model, not any specific training record
- **Record reconstruction** (harder): recovers something close to a specific training record; requires very overfit models or white-box weight access
- **Verbatim extraction** (LLMs): large models can reproduce training text verbatim through sampling — different mechanism, same privacy concern

**Defense priority order:**
1. Don't expose confidence scores in your API — return class labels only
2. Rate limit queries to detect inversion attempts
3. Add output perturbation if confidence scores must be returned
4. DP training for formal guarantees (notebook 14)

**Next:** Notebook 07 covers attribute inference — inferring sensitive attributes (race, health status, income) that weren't used as model inputs, purely from model outputs on related queries. Distinct from both membership inference and model inversion, but related to both.